# CAM-ConvLSTM End-to-End Training Pipeline
This notebook demonstrates loading real gridded data, preprocessing it, and training the CAM-ConvLSTM model end-to-end.

In [1]:
import sys
import os
import pandas as pd

# Add project root to Python path to find the 'src' directory
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Added project root to sys.path: {project_root}")

# Import your new ConvLSTM pipeline class
from src.improved_convlstm_multitask_pipeline import MultitaskConvLSTMPipeline

Added project root to sys.path: c:\Users\peera\Desktop\DroughtLSTM_oneday


c:\Users\peera\.conda\envs\drought_lstm_base\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch, PyTorch Lightning, and Optuna successfully imported.
ConvLSTM Pipeline: Successfully imported utility functions.


c:\Users\peera\.conda\envs\drought_lstm_base\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.1 is exactly one major version older than the runtime version 6.31.1 at api.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(


In [2]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from pytorch_lightning import Trainer
from src.data_utils import load_and_prepare_data, split_data_chronologically
from src.feature_utils import engineer_features
from src.preprocess_utils import scale_data
from src.cam_clstm.causal_clsm_model import MyConvLSTMModel
from src.cam_clstm.causal_multitask_lightning_module import CausalMultitaskLightningModule
from src.cam_clstm.causual_clstm_pipeline import GriddedConvLSTMDataset


In [3]:
%pwd

'c:\\Users\\peera\\Desktop\\DroughtLSTM_oneday\\notebooks'

In [4]:
temporal_features = ['soi', 'dmi', 'pdo', 'nino4', 'nino34', 'nino3']

def extract_temporal_only_tensor(df, temporal_features, config):
    """
    Returns a tensor of shape [B, T, C_temp] where:
    - B = number of spatial samples (same as X.shape[0])
    - T = sequence length
    - C_temp = number of temporal-only features
    """
    df_temp = df.copy()
    time_col = config['data']['time_column']

    # Group by time and take the mean across locations (should all be same if global)
    df_temp = df_temp.groupby(time_col)[temporal_features].mean().reset_index()

    # Sort time
    df_temp = df_temp.sort_values(time_col)

    # Convert to tensor: [T, C_temp]
    temporal_only = torch.tensor(df_temp[temporal_features].values, dtype=torch.float32)

    return temporal_only



In [5]:
config = {
    'feature_names': ['pre', 'pet', 'tmp', 'cld', 'dmi', 'soi', 'nino3', 'nino4', 'nino34', 'pdo'],
    'grid': {'height': 10, 'width': 10},
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'training': {
        'epochs': 50,
        'learning_rate': 1e-3   # ✅ add this line
    },
    'data': {
        'data_path': '../data/processed/full_scaled.csv',
        'train_end_date': '2017-12-31',
        'validation_end_date': '2020-12-31',
        'time_column': 'time',
        'lat_column': 'lat',
        'lon_column': 'lon',
        'features_to_grid': ['tmp', 'dtr', 'cld', 'tmx', 
      'tmn', 'wet', 'vap', 'soi','pre', 'pet',"spei" ]
    },
}

# Load CSV directly
df = pd.read_csv('../data/processed/full_scaled.csv')
df_train, df_val, df_test = split_data_chronologically(df, config)

Splitting data: Train ends 2017-12-31 00:00:00, Validation ends 2020-12-31 00:00:00
Train set shape: (251316, 19), Time range: 1901-01-16 00:00:00 to 2017-12-16 00:00:00
Validation set shape: (6444, 19), Time range: 2018-01-16 00:00:00 to 2020-12-16 00:00:00
Test set shape: (6444, 19), Time range: 2021-01-16 00:00:00 to 2023-12-16 00:00:00


In [11]:
from src.cam_clstm.convert_csv_to_lstm_dataset import convert_csv_to_lstm_dataset

X_train, y_train_dict, mask_train = convert_csv_to_lstm_dataset(df_train, config, seq_len=12)
X_val,   y_val_dict,   mask_val   = convert_csv_to_lstm_dataset(df_val, config, seq_len=12)
X_test,  y_test_dict,  mask_test  = convert_csv_to_lstm_dataset(df_test, config, seq_len=12)


--- Starting Data Gridding Process (Fixed Step Method) ---
Using fixed grid step of: 0.5 degrees
Grid boundaries: LAT (6.25, 20.25), LON (97.75, 105.25)
Calculated grid dimensions: Height=29, Width=16
Created 2D validity mask (29x16) with 179 valid data pixels.
Pivoting data into a 4D tensor of shape (1404, 29, 16, 11)...


c:\Users\peera\Desktop\DroughtLSTM_oneday\src\grid_utils.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['row_idx'] = ((df[lat_col] - lat_min) / fixed_step).round().astype(int)
c:\Users\peera\Desktop\DroughtLSTM_oneday\src\grid_utils.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['col_idx'] = ((df[lon_col] - lon_min) / fixed_step).round().astype(int)


--- Data Gridding Process Finished ---
Created sequence data: X=(1392, 12, 10, 29, 16), y=(1392, 1, 29, 16)
--- Starting Data Gridding Process (Fixed Step Method) ---
Using fixed grid step of: 0.5 degrees
Grid boundaries: LAT (6.25, 20.25), LON (97.75, 105.25)
Calculated grid dimensions: Height=29, Width=16
Created 2D validity mask (29x16) with 179 valid data pixels.
Pivoting data into a 4D tensor of shape (36, 29, 16, 11)...
--- Data Gridding Process Finished ---
Created sequence data: X=(24, 12, 10, 29, 16), y=(24, 1, 29, 16)
--- Starting Data Gridding Process (Fixed Step Method) ---
Using fixed grid step of: 0.5 degrees
Grid boundaries: LAT (6.25, 20.25), LON (97.75, 105.25)
Calculated grid dimensions: Height=29, Width=16
Created 2D validity mask (29x16) with 179 valid data pixels.
Pivoting data into a 4D tensor of shape (36, 29, 16, 11)...
--- Data Gridding Process Finished ---
Created sequence data: X=(24, 12, 10, 29, 16), y=(24, 1, 29, 16)


c:\Users\peera\Desktop\DroughtLSTM_oneday\src\grid_utils.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['row_idx'] = ((df[lat_col] - lat_min) / fixed_step).round().astype(int)
c:\Users\peera\Desktop\DroughtLSTM_oneday\src\grid_utils.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['col_idx'] = ((df[lon_col] - lon_min) / fixed_step).round().astype(int)
c:\Users\peera\Desktop\DroughtLSTM_oneday\src\grid_utils.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from

In [14]:
temporal_tensor_train = extract_temporal_only_tensor(df_train, 
                                                     temporal_features=['soi', 'dmi', 'pdo', 'nino4', 
                                                                        'nino34', 'nino3'],config=config)
temporal_tensor_val = extract_temporal_only_tensor(df_val,
                                                   temporal_features=['soi', 'dmi', 'pdo', 'nino4', 
                                                                      'nino34', 'nino3'],config=config)
temporal_tensor_test = extract_temporal_only_tensor(df_test,
                                                    temporal_features=['soi', 'dmi', 'pdo', 'nino4', 
                                                                       'nino34', 'nino3'],config=config)

In [17]:
temporal_tensor_train.shape

torch.Size([1404, 6])

In [15]:
train_dataset = GriddedConvLSTMDataset(X_train, y_train_dict, temporal_tensor_train)
val_dataset   = GriddedConvLSTMDataset(X_val,   y_val_dict,   temporal_tensor_val)
test_dataset  = GriddedConvLSTMDataset(X_test,  y_test_dict,  temporal_tensor_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8)
test_loader  = DataLoader(test_dataset, batch_size=8)


In [16]:
model = MyConvLSTMModel(
    input_channels=X_train.shape[2],
    height=X_train.shape[3],
    width=X_train.shape[4],
    hidden_channels=32,
    use_pos_enc=True,
    use_spatial_attn=True,
    use_temporal_only=True,
)

lightning_module = CausalMultitaskLightningModule(model, config)
trainer = Trainer(max_epochs=config['training']['epochs'])
trainer.fit(lightning_module, train_loader, val_loader)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type            | Params | Mode 
----------------------------------------------------
0 | model   | MyConvLSTMModel | 56.3 K | train
1 | loss_fn | MSELoss         | 0      | train
----------------------------------------------------
56.3 K    Trainable params
0         Non-trainable params
56.3 K    Total params
0.225     Total estimated model params size (MB)
33        Modules in train mode
0         Modules in eval mode


Initializing MyConvLSTMModel with input channels: 10
Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]temporal_only_input shape: torch.Size([8, 6])


C:\Users\peera\AppData\Roaming\Python\Python312\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


RuntimeError: Sizes of tensors must match except in dimension 2. Expected size 12 but got size 6 for tensor number 1 in the list.